In [30]:
import csv
import math

rows = []
with open('dataset_phishing.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

feature_cols = [c for c in rows[0].keys() if c not in ('url', 'status')]
n = len(rows)
print(n, "rows,", len(feature_cols), "features")

11430 rows, 87 features


In [31]:
def variance(col_name):
    vals = [float(r[col_name]) for r in rows]
    m = sum(vals) / n
    return sum((v - m) ** 2 for v in vals) / n

In [32]:
feature_cols = [col for col in feature_cols if variance(col) > 0]
print("After zero variance drop:", len(feature_cols))

After zero variance drop: 81


In [33]:
def dominant_pct(col_name):
    vals = [float(r[col_name]) for r in rows]
    most_common = max(set(vals), key=vals.count)
    return vals.count(most_common) / n

feature_cols = [col for col in feature_cols
                if not (len(set(float(r[col]) for r in rows)) == 2
                        and dominant_pct(col) >= 0.98)]
print("After near-zero variance drop:", len(feature_cols))

After near-zero variance drop: 67


In [34]:
for col in feature_cols:
    vals = [float(r[col]) for r in rows]
    if -1 in vals:
        valid  = sorted(v for v in vals if v != -1)
        median = valid[len(valid) // 2]
        for r in rows:
            if float(r[col]) == -1:
                r[col] = str(median)
        print(col, "— fixed with median:", median)

domain_registration_length — fixed with median: 245.0
domain_age — fixed with median: 5056.0


In [35]:
for col in feature_cols:
    vals   = [float(r[col]) for r in rows]
    m      = sum(vals) / n
    median = sorted(vals)[n // 2]
    if median != 0 and m / median > 2.0:
        for r in rows:
            r[col] = str(math.log(float(r[col]) + 1))
        print(col, "— log transformed")

nb_hyperlinks — log transformed
ratio_extHyperlinks — log transformed
ratio_intMedia — log transformed
domain_registration_length — log transformed
web_traffic — log transformed


In [36]:
drop_corr = []

for i in range(len(feature_cols)):
    col_a = feature_cols[i]
    if col_a in drop_corr:
        continue
    a  = [float(r[col_a]) for r in rows]
    ma = sum(a) / n
    for j in range(i + 1, len(feature_cols)):
        col_b = feature_cols[j]
        if col_b in drop_corr:
            continue
        b  = [float(r[col_b]) for r in rows]
        mb = sum(b) / n
        num   = sum((x - ma) * (y - mb) for x, y in zip(a, b))
        den_a = math.sqrt(sum((x - ma) ** 2 for x in a))
        den_b = math.sqrt(sum((y - mb) ** 2 for y in b))
        if den_a == 0 or den_b == 0:
            continue
        if abs(num / (den_a * den_b)) > 0.75:
            drop_corr.append(col_b)
            print(col_a, "↔", col_b, "— dropping", col_b)

feature_cols = [c for c in feature_cols if c not in drop_corr]
print("After correlation drop:", len(feature_cols))

length_url ↔ length_words_raw — dropping length_words_raw
ip ↔ ratio_digits_url — dropping ratio_digits_url
nb_and ↔ nb_eq — dropping nb_eq
shortest_word_host ↔ avg_word_host — dropping avg_word_host
longest_words_raw ↔ longest_word_path — dropping longest_word_path
longest_words_raw ↔ avg_words_raw — dropping avg_words_raw
ratio_intHyperlinks ↔ links_in_tags — dropping links_in_tags
After correlation drop: 60


In [37]:
target = [1 if r['status'] == 'phishing' else 0 for r in rows]

def pbc(col_name):
    vals   = [float(r[col_name]) for r in rows]
    m      = sum(vals) / n
    g1     = [v for v, t in zip(vals, target) if t == 1]
    g0     = [v for v, t in zip(vals, target) if t == 0]
    std    = math.sqrt(sum((v - m) ** 2 for v in vals) / n)
    if std == 0:
        return 0
    return abs((sum(g1)/len(g1) - sum(g0)/len(g0)) / std * math.sqrt(len(g1) * len(g0) / (n * n)))

feature_cols = [col for col in feature_cols if pbc(col) >= 0.05]
print("After PBC drop:", len(feature_cols))

After PBC drop: 48


In [38]:
for col in feature_cols:
    vals = [float(r[col]) for r in rows]
    lo, hi = min(vals), max(vals)
    for r in rows:
        r[col] = str(round((float(r[col]) - lo) / (hi - lo), 6) if hi != lo else 0.0)
print("All features normalised to [0, 1]")

All features normalised to [0, 1]


In [39]:
output_cols = ['url'] + feature_cols + ['status']

with open('dataset_processed.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=output_cols, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(rows)

print("Saved: dataset_processed.csv")
print("Rows:", n, "| Features:", len(feature_cols))

Saved: dataset_processed.csv
Rows: 11430 | Features: 48
